In [2]:
import sys
from pathlib import Path

# Add parent directory of "a" to sys.path
sys.path.append(str(Path.cwd().parent))

In [3]:
import numpy as np
import pandas as pd
from CODECbreakCode.AudioMixer import FullTrackAudioMixer
import CODECbreakCode.Evaluator as Evaluator
from CODECbreakCode.Evaluator import MeasureHAAQIOutput
import argparse
from CODECbreakCode.Optimiser.config import get_config, normalize_action, denormalize_action

In [5]:
import warnings
warnings.filterwarnings("ignore", "Possible clipped samples in output.")
warnings.filterwarnings("ignore",message="Warning: input samples dtype is np.float64. Converting to np.float32")

## Loading Beatles Prject

In [6]:
Beatles_Mixing_Path = str(Path("../AudioEX/Beatles").resolve())
Beatles_Noise_Generator_MP3 = FullTrackAudioMixer(Beatles_Mixing_Path)

Vocal duration orginal is 33.418166666666664 seconds, now is the 8.0, the audio changing to the MONO
Drum duration orginal is 33.418166666666664 seconds, now is the 8.0, the audio changing to the MONO
Bass duration orginal is 33.418166666666664 seconds, now is the 8.0, the audio changing to the MONO
Other duration orginal is 33.418166666666664 seconds, now is the 8.0,  the audio changing to the MONO
Mixing File Load Sucessful


In [7]:
Beatles_Referece_File = Beatles_Noise_Generator_MP3.TestDynNoisedFullTrack([0]*28,"Reference_IN_FULL.wav",isNormalised=False,isCompensated=True)
print(f"Referece_File:{Beatles_Referece_File}")

Referece_MP3File = Evaluator.Mp3LameLossyCompress(Beatles_Referece_File,64)
print(f"Beatles_Referece_Mp3File:{Referece_MP3File}")

Referece_File:/home/codecrack/CodecBreakerwithRL/Paper_Code_Framework_for_Degradator/AudioEX/Beatles/Mixing_Result/Reference_IN_FULL.wav
Beatles_Referece_Mp3File:/home/codecrack/CodecBreakerwithRL/Paper_Code_Framework_for_Degradator/AudioEX/Beatles/Mixing_Result_Mp3_Wav/Reference_IN_FULL_64kbps.wav


In [8]:
####initalise the Haaqi
MeasureHAAQI = MeasureHAAQIOutput(Referece_MP3File)#Initilize the HAAQI with a permanent reference
MeasureHAAQI.MeasureHAQQIOutput(Referece_MP3File) #Test on how far from itself to itself

0.995047379354648

In [9]:
import tempfile
import os
BASE_DIR = Beatles_Mixing_Path + "/Mixing_Result"
TMP_SUBDIR = os.path.join(BASE_DIR, "tmp")

# make sure it exists once at startup:
os.makedirs(TMP_SUBDIR, exist_ok=True)
def haaqi_reward_muti_fn(solution: np.ndarray, is_normalised=True) -> float:
    if is_normalised:
        solution = denormalize_action(solution)
#    print(f'solution:{solution}, the type is {type(solution), type(solution[0])}')
 
    # Create a unique temp‐file name
    fd, degradated_filename = tempfile.mkstemp(prefix="dynC_", suffix=".wav")
    os.close(fd)  # we’ll let your compressor write to that path

    try:
        solution = list(solution)
        gener_Audio = Beatles_Noise_Generator_MP3.TestDynNoisedFullTrack(solution,degradated_filename,isNormalised=False,isCompensated=True)
        gener_Audio_mp3 = Evaluator.Mp3LameLossyCompress(gener_Audio, 64)
        score = MeasureHAAQI.MeasureHAQQIOutput(gener_Audio_mp3)
    finally:
        # clean up
        try:
            os.remove(degradated_filename)
        except OSError:
            pass

    return round(1 - score, 4)

## Optimiser Strategy

### MC

In [ ]:
from CODECbreakCode.Optimiser.mc_opt_multi import MonteCarloMultiOptimiser
from CODECbreakCode.Optimiser.config import get_config
cfg = get_config()

In [ ]:
mc = MonteCarloMultiOptimiser(reward_fn = haaqi_reward_muti_fn,solution_dim=28, low = cfg["env"]["act_min"],high = cfg["env"]["act_min"])
per_step_bests = mc.optimise_by_steps(n_steps=10, sampling_per_step=60)

In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',
              'Hum','Hiss','CL','Thre','Ratio','Attk','Release',
              'Hum','Hiss','CL','Thre','Ratio','Attk','Release',
              'Hum','Hiss','CL','Thre','Ratio','Attk','Release',
             ]
MC_Data, MC_Data_Full = mc.save_results(filefold=Beatles_Mixing_Path, para_columns=para_columns, is_outputfulldata = True)

### GA

In [ ]:
import CODECbreakCode.Optimiser.genetic_evo
from CODECbreakCode.Optimiser.genetic_evo import GeneticOptimiser

In [ ]:
GA_Opt = GeneticOptimiser()
GA_Opt.ga_init_env()
GA_Opt.set_fitnessfun(haaqi_reward_muti_fn)

In [ ]:
GA_Opt.run(num_generations = 10,sol_per_pop = 60)

In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',
              'Hum','Hiss','CL','Thre','Ratio','Attk','Release',
              'Hum','Hiss','CL','Thre','Ratio','Attk','Release',
              'Hum','Hiss','CL','Thre','Ratio','Attk','Release',
             ]
Evo_Data, Evo_Data_Full = GA_Opt.save_results(filefold=Beatles_Mixing_Path, para_columns=para_columns, is_outputfulldata = True)

### RL

In [4]:
from CODECbreakCode.Optimiser.continous_RL_train import continous_RL_train as CRLTrain

2026-06-04 16:06:39.202976: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-04 16:06:39.229842: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-04 16:06:39.229869: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-04 16:06:39.231323: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-04 16:06:39.237541: I tensorflow/core/platform/cpu_feature_guar

In [ ]:
trainner = CRLTrain(sub_episode_length=11, sub_episode_num_single_batch=3, env_num=1)
trainner.set_environments(haaqi_reward_muti_fn)
trainner.train(update_num=21, eval_intv=5)

In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',
              'Hum','Hiss','CL','Thre','Ratio','Attk','Release',
              'Hum','Hiss','CL','Thre','Ratio','Attk','Release',
              'Hum','Hiss','CL','Thre','Ratio','Attk','Release',
             ]
RL_data, RL_F_data = trainner.save_results(filefold=Beatles_Mixing_Path, para_columns=para_columns,is_outputfulldata = True)

### Cleanning the File

In [ ]:
Beatles_Noise_Generator_MP3.EraseTheMixing()